In [1]:
queries = [
    # Easy: keyword-based
    {"id": 1, "query": "argan oil hair spray", "type": "keyword"},
    {"id": 2, "query": "volumizing dry shampoo", "type": "keyword"},
    {"id": 3, "query": "sulfate free shampoo", "type": "keyword"},

    # Medium: semantic
    {"id": 4, "query": "something to make frizzy hair smooth", "type": "semantic"},
    {"id": 5, "query": "product that protects hair from heat damage", "type": "semantic"},
    {"id": 6, "query": "gentle cleanser for sensitive scalp", "type": "semantic"},
    {"id": 7, "query": "natural scent that is not overpowering", "type": "semantic"},

    # Complex: require context, filtering, or reasoning
    {"id": 8, "query": "best leave-in conditioner for thick curly hair under $20", "type": "complex"},
    {"id": 9, "query": "what do people say about this product causing hair loss", "type": "complex"},
    {"id": 10, "query": "highly rated hair product that works for both men and women", "type": "complex"},
]

for q in queries:
    print(f"[{q['type'].upper():8}] {q['query']}")

[KEYWORD ] argan oil hair spray
[KEYWORD ] volumizing dry shampoo
[KEYWORD ] sulfate free shampoo
[SEMANTIC] something to make frizzy hair smooth
[SEMANTIC] product that protects hair from heat damage
[SEMANTIC] gentle cleanser for sensitive scalp
[SEMANTIC] natural scent that is not overpowering
[COMPLEX ] best leave-in conditioner for thick curly hair under $20
[COMPLEX ] what do people say about this product causing hair loss
[COMPLEX ] highly rated hair product that works for both men and women


In [16]:
#  BM25 
from rank_bm25 import BM25Okapi
import nltk
import pandas as pd
nltk.download('punkt')
from nltk.tokenize import word_tokenize
import pickle
result = pd.read_parquet('../data/processed/merged.parquet')

# Build corpus: combine review title + text
#corpus = (result['title'].fillna('') + ' ' + result['text'].fillna('')).tolist()
#tokenized_corpus = [word_tokenize(doc.lower()) for doc in corpus]
#bm25 = BM25Okapi(tokenized_corpus)

with open('../models/bm25_model.pkl', 'rb') as f:
    bm25 = pickle.load(f)

with open('../data/processed/tokenized_corpus.pkl', 'rb') as f:
    tokenized_corpus = pickle.load(f)


[nltk_data] Downloading package punkt to /Users/arafat/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [17]:
def bm25_search(query, top_k=5):
    tokens = word_tokenize(query.lower())
    scores = bm25.get_scores(tokens)
    top_indices = scores.argsort()[::-1][:top_k]
    result['score'] = scores  # add scores to result for display
    return result.iloc[top_indices][['product_title', 'text', 'rating', 'score']].reset_index(drop=True)

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

# Load pre-computed embeddings instead of re-encoding
corpus_embeddings = np.load("../data/processed/new_embeddings.npy")

print(f"Embeddings shape: {corpus_embeddings.shape}")  # should be (num_docs, 384)



Embeddings shape: (701528, 384)


In [5]:
import faiss

dimension = corpus_embeddings.shape[1]  # 384 for all-MiniLM-L6-v2

index = faiss.IndexFlatL2(dimension)   # L2 (Euclidean) distance
# or use cosine similarity:
# index = faiss.IndexFlatIP(dimension) # Inner product (cosine if normalized)

index.add(corpus_embeddings)
print(f"Total vectors indexed: {index.ntotal}")

Total vectors indexed: 701528


In [6]:
def semantic_search(query, top_k=5):
    query_embedding = model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    
    results = result.iloc[indices[0]].copy()
    results['distance'] = distances[0]
    
    return pd.DataFrame(results)[['product_title', 'text', 'rating', 'distance']].reset_index(drop=True)

In [14]:
semantic_search("sunscreen that doesn't make my hair greasy")

,product_title,text,rating,distance
0,"Spray Bottles Travel Size, AMMAX 4pcs Fine Mis...",Shape of top of bottle actually makes my finge...,3.0,0.530763
1,CAITLYN Afro Kinky Curly Lace Front Wigs Human...,The wig was cute and curls were as expected. S...,3.0,0.545138
2,BESSKY 1 Pair/2PCS Style Exquisite Gold Bee Ha...,Super cute but too heavy for really fine thin ...,5.0,0.579942
3,REFINE - Germany - Pedicure Corn Plane Callus ...,I wish i had a before picture of my heels. I'v...,5.0,0.591899
4,NPW Unicorn Ouch Bandages,I have a little relative who absolutely loves ...,5.0,0.597433


In [27]:
all_results = {}

for q in queries:
    qid = q['id']
    query_text = q['query']
    all_results[qid] = {
        "query": query_text,
        "type": q['type'],
        "bm25": bm25_search(query_text),
        "semantic": semantic_search(query_text)
    }
    print(f"Done: Q{qid} — {query_text}")

Done: Q1 — argan oil hair spray
Done: Q2 — volumizing dry shampoo
Done: Q3 — sulfate free shampoo
Done: Q4 — something to make frizzy hair smooth
Done: Q5 — product that protects hair from heat damage
Done: Q6 — gentle cleanser for sensitive scalp
Done: Q7 — natural scent that is not overpowering
Done: Q8 — best leave-in conditioner for thick curly hair under $20
Done: Q9 — what do people say about this product causing hair loss
Done: Q10 — highly rated hair product that works for both men and women


In [28]:
chosen = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # adjust to your chosen 5

for qid in chosen:
    q = all_results[qid]
    print(f"\n{'='*60}")
    print(f"Q{qid} [{q['type'].upper()}]: {q['query']}")
    print(f"\n--- BM25 Top 5 ---")
    display(q['bm25'])
    print(f"\n--- Semantic Top 5 ---")
    display(q['semantic'])


Q1 [KEYWORD]: argan oil hair spray

--- BM25 Top 5 ---


,product_title,text,rating,score
0,"Nadira Organics Virgin Argan Oil for Skin, Fac...","Love this Argan oil, on my second bottle. Use ...",5.0,20.852335
1,Savanaahs Organic Argan Oil 4 Oz / 120 ML 100%...,I love using Argan oil in my hair it is the pe...,4.0,20.762904
2,Savanaahs Organic Argan Oil 4 Oz / 120 ML 100%...,I've been fooled by the 'Argan oil' label befo...,5.0,20.295554
3,Livordo Moroccan Argan Oil Essential Organic C...,My daughter takes every Argan oil I get from m...,5.0,19.216008
4,Savanaahs Organic Argan Oil 4 Oz / 120 ML 100%...,I love this oil. It is perfect. I used it on...,5.0,19.038683



--- Semantic Top 5 ---


,product_title,text,rating,score
0,Cuticle Nippers,The cuticle nipper kept getting stuck. It does...,1.0,0.827026
1,EcoBrow - All Natural Eyebrow Defining Wax (Pe...,"I'm in love, I received the Liz color as a sam...",5.0,0.781909
2,New York Biology Vitamin C Serum for Face and ...,"Well, they weren't lying when they said it wor...",5.0,0.769094
3,Compatible Oral-B Replacement Brush Heads - Va...,Easy Replacement and worked on other device I ...,5.0,0.768997
4,WAVEBUILDER Seamless Durag- Black,I no longer have to worry about a line going d...,5.0,0.768802



Q2 [KEYWORD]: volumizing dry shampoo

--- BM25 Top 5 ---


...

,product_title,text,rating,score
0,"Fleece Ear Warmers Headband for Men & Women, T...",Made my face break out..,2.0,0.697477
1,Mitchell's Wool Fat Shave Refill Soap,I purchased this shave soap based on the reput...,2.0,0.696505
2,Salux Nylon Japanese Beauty Skin Bath Wash Clo...,I had a Korean body scrub in Los Angeles and f...,5.0,0.684296
3,Moresoo 12 Inch Clip in Human Hair Extensions ...,Definitely will keep ordering from this seller...,5.0,0.678040
4,Bath Scrubber Body Brush Shower Scrubber Back ...,"Loved the brush because of its softness, but t...",4.0,0.675264



Q10 [COMPLEX]: highly rated hair product that works for both men and women

--- BM25 Top 5 ---


,product_title,text,rating,score
0,JOOYHOOM Professional Hair Cutting Scissors Se...,Great price for a set of hair cutting kit! Wor...,5.0,36.373014
1,SilkFeet NightCare Flexible Bladeless Exfoliat...,BEST BEST BEST foot care product,5.0,30.565780
2,Baby Fresh Powder Perfume Scent - .33oz/10ml R...,The best thing I’ve ever smelled men and women...,5.0,29.566826
3,Fearless Tape - Sensitive Skin - Women's Doubl...,Good for both men and women.,5.0,29.356378
4,Cuba Quad 1 Gift Set (4 x 1.17 fl.oz. Eau De T...,Amazing combination of scents and flavors. Uni...,5.0,28.508610



--- Semantic Top 5 ---


,product_title,text,rating,score
0,"Gel Toe Separators,XUZOU Toe Straighteners for...",They arrived very quickly and I used them righ...,5.0,0.785000
1,Organic Activated Black Best Charcoal Natural ...,When you’re looking for a teeth whitener that’...,5.0,0.744655
2,Healthcom Makeup Bags and Cases,This is a good buy. I'm not sure what to call ...,4.0,0.739447
3,Francis,"Ok fragrance, but too floral for me. I would ...",3.0,0.739447
4,Hot Spike Stud Blouse Shirts Collar Neck Tip B...,Cute fell in love with it but just took a long...,5.0,0.735885


In [ ]:
result.info()

<class 'pandas.DataFrame'>
RangeIndex: 701528 entries, 0 to 701527
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             701528 non-null  float64
 1   title              701528 non-null  str    
 2   text               701528 non-null  str    
 3   verified_purchase  701528 non-null  bool   
 4   product_title      701528 non-null  str    
 5   average_rating     701528 non-null  float64
 6   price              701528 non-null  str    
 7   description        701528 non-null  object 
 8   store              651636 non-null  str    
 9   details            701528 non-null  str    
dtypes: bool(1), float64(2), object(1), str(6)
memory usage: 414.8+ MB


In [7]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [8]:
load_dotenv()  # Load environment variables from .env file

def load_llm():
    """Instantiate and return the LLaMA-3 chat model via HuggingFace endpoint."""

    llm_endpoint = HuggingFaceEndpoint(
        repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
        task="text-generation",
        max_new_tokens=300,
        provider="auto"
    )
    return ChatHuggingFace(llm=llm_endpoint)

In [19]:
def retrieve_context(query: str, top_k: int = 5) -> str:
    """
    Calls the Milestone 1 semantic_search and formats the results
    into a plain text context block for the LLM prompt.
    """
    results = semantic_search(query, top_k=5)

    context_parts = []
    for i, row in results.iterrows():
        part = (
            f"Product: {row['product_title']}\n"
            f"Review: {row['text']}\n"
            f"Rating: {row['rating']}\n"
            f"Similarity distance: {row['distance']:.4f}"
        )
        context_parts.append(part)

    return "\n\n---\n\n".join(context_parts)

In [10]:
PROMPT_TEMPLATE = """
You are a helpful product recommendation assistant.
Use ONLY the product information provided below to answer the user's question.
If the products do not match the query well, say so honestly.

Retrieved Products:
{context}

User Question: {question}

Your Answer:
"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)


# 4. RAG PIPELINE

def build_rag_pipeline():
    """Build and return a LangChain RAG chain: retrieve context, fill prompt, call LLM."""

    llm = load_llm()

    # Chain: retrieve context -> fill prompt -> call LLM
    chain = (
        {
            "context": lambda x: retrieve_context(x["question"]),
            "question": RunnablePassthrough() | (lambda x: x["question"])
        }
        | prompt
        | llm
    )

    return chain


def rag_query(question: str, top_k: int = 5) -> str:
    """
    Main entry point. Pass a natural language product question,
    get a grounded answer back.
    """
    llm = load_llm()

    # Retrieve context from your FAISS index
    context = retrieve_context(question, top_k=top_k)

    # Fill the prompt
    filled_prompt = prompt.invoke({
        "context": context,
        "question": question
    })

    # Call the LLM
    response = llm.invoke(filled_prompt)
    return response.content


In [ ]:
rag_query("What do people say about this product causing hair loss?", top_k=5)

'Based on the product information provided, I do not have information about hair loss related to the products listed. However, one product, "Brazilian Virgin Hair Body Wave 3 Bundles with Closure 10A Grade 100 Unprocessed Human Hair Weave Extensions Bundles with 4×4 Three Part Lace Closures Brazilian Hair With Closure (18 20 22+16)" is related to hair. Unfortunately, it does not have any customer reviews mentioning hair loss.'

### Testing different prompts using meta-llama/Meta-Llama-3-8B-Instruct

In [26]:
queries = [
    'gentle cleanser for sensitive scalp',
    'natural scent that is not overpowering',
    'best leave-in conditioner for thick curly hair under $20',
    'what do people say about this product causing hair loss',
    'highly rated hair product that works for both men and women'
]

queries

['gentle cleanser for sensitive scalp',
 'natural scent that is not overpowering',
 'best leave-in conditioner for thick curly hair under $20',
 'what do people say about this product causing hair loss',
 'highly rated hair product that works for both men and women']

In [30]:
print("\n\n=== RAG Query Results using meta-llama/Meta-Llama-3-8B-Instruct ===\n")
for q in queries:
    print(rag_query(q, top_k=3), "\n" + "="*50 + "\n")



=== RAG Query Results using meta-llama/Meta-Llama-3-8B-Instruct ===

Based on the products provided, I don't see any direct match for a gentle cleanser for a sensitive scalp. The products listed include hair accessories, skin care sets, synthetic braids, lip care, and a tanning lotion, but none of them are specifically a gentle cleanser.

However, if you're looking for a gentle skin care product that might also be suitable for a sensitive scalp, the Drunk Elephant Rise + Glow Duo could be a possibility, as it's a gentle skin care set that includes a vitamin C serum and a hydrating gel. 

Based on the provided products, I couldn't find any information related to a product with a "natural scent that is not overpowering". The products listed include a digital timer, plastic pump bottles, hair accessories, and hair care tools, but none of them mention scent or fragrance. If you're looking for a product with a natural scent, I recommend searching for products specifically labeled as "natu

### Testing different prompts using "mistralai/Mistral-7B-Instruct-v0.2"

In [35]:
def load_llm():
    """Instantiate and return the LLaMA-3 chat model via HuggingFace endpoint."""

    llm_endpoint = HuggingFaceEndpoint(
        repo_id="mistralai/Mistral-7B-Instruct-v0.2",
        task="text-generation",
        max_new_tokens=300,
        provider="auto"
    )
    return ChatHuggingFace(llm=llm_endpoint)

In [36]:
print("\n\n=== RAG Query Results using mistralai/Mistral-7B-Instruct-v0.2 ===\n")
for q in queries:
    print(rag_query(q, top_k=3), "\n" + "="*50 + "\n")



=== RAG Query Results using mistralai/Mistral-7B-Instruct-v0.2 ===

Based on the information provided, I would recommend the Aveda Lip Saver, despite it being a lip product rather than a cleanser for the scalp. The reason for my recommendation is that the user mentioned in their review that they love Aveda products, and they were disappointed with the overpricing of the Aveda Lip Saver by the seller. However, the user did not mention anything about the product itself being suitable or not for sensitive scalps. Therefore, I cannot make a definitive recommendation based on the product information provided alone. If you have any additional information or context that could help narrow down the options, please let me know and I'll do my best to help you out!

I hope this helps answer your question, and I apologize for the lengthy response. I want to make sure that I provide you with accurate and useful information, rather than just giving you a quick answer that might not be entirely rel